# Tests de Hipótesis en Python
## Referencia estadística — Clase 11 · TPAD · FCE-UBA

Este notebook es una **guía de referencia** para los principales tests de hipótesis
paramétricos, con énfasis en su implementación en Python (`scipy.stats`, `statsmodels`).

Cada sección incluye:
- Definición formal e hipótesis
- Fórmula del estadístico con su distribución muestral
- Región crítica y criterio p-valor para los tres tipos de H₁
- Código ejecutable con ejemplos sobre datos reales del Merval
- Cuadro comparativo con las funciones del profesor (donde corresponda)

---

## Tabla de contenidos

| # | Test | Estadístico | Distribución | Función profesor |
|---|---|---|---|---|
| 0 | Marco conceptual | — | — | — |
| 1 | z — media (σ conocida) | z_obs | N(0,1) | — |
| **2** | **t — media (σ desconocida)** | t_obs | t(n−1) | `TH_MEDIA_VARDESCON` ✅ |
| **3** | **χ² — varianza** | χ²_obs | χ²(n−1) | `TH_MEDIA_VARIANZA` ✅ |
| 4 | z — proporción | z_obs | N(0,1) | — |
| 5 | t — diferencia de medias | t_obs | t(ν) | — |
| 6 | F — cociente de varianzas | F_obs | F(ν₁,ν₂) | — |

> Las funciones del profesor están definidas en `TPAD_Clase_11.ipynb` (misma carpeta)
> y cubren los tests 2 y 3.

In [1]:
"""
Imports necesarios para todos los tests del notebook.
Se importan una sola vez; las celdas posteriores los reutilizan.
"""
import numpy as np
import pandas as pd
import scipy.stats as ss
from statsmodels.stats.proportion import proportions_ztest
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
"""
Carga de datos del Merval (^MERV) para el período 2025-01-01 a 2025-12-01.
Estos datos se usan como conjunto de ejemplo en todas las secciones.

Variables generadas
-------------------
df      : DataFrame con columnas 'Fecha' y 'Precio'
precios : Series  — precios de cierre (sin NaN)
n       : int     — cantidad de observaciones
media   : float   — media muestral (x̄)
S       : float   — desvío estándar muestral (ddof=1)
sem1    : Series  — precios primer semestre  (ene–jun 2025)
sem2    : Series  — precios segundo semestre (jul–nov 2025)
"""
merval = yf.download("^MERV", start="2025-01-01", end="2025-12-01", progress=False)
df     = merval[["Close"]].reset_index()
df.columns = ["Fecha", "Precio"]

precios = df["Precio"].dropna()
n       = len(precios)
media   = float(precios.mean())
S       = float(precios.std())   # ddof=1 por defecto en pandas

# Partición por semestre (para tests de dos muestras en secciones 5 y 6)
sem1 = df[df["Fecha"] <  "2025-07-01"]["Precio"].dropna()
sem2 = df[df["Fecha"] >= "2025-07-01"]["Precio"].dropna()

print(f"Período : {df['Fecha'].min().date()} → {df['Fecha'].max().date()}")
print(f"n       = {n} días hábiles")
print(f"x̄       = ${media:,.2f}")
print(f"S       = ${S:,.2f}")
print(f"sem1 : n={len(sem1)}  (ene–jun 2025)")
print(f"sem2 : n={len(sem2)}  (jul–nov 2025)")

Período : 2025-01-02 → 2025-11-28
n       = 223 días hábiles
x̄       = $2,271,495.74
S       = $327,523.45
sem1 : n=118  (ene–jun 2025)
sem2 : n=105  (jul–nov 2025)


---
## 0 · Marco conceptual

### Definiciones básicas

| Símbolo | Nombre | Descripción |
|---|---|---|
| H₀ | Hipótesis nula | Afirmación que se asume verdadera hasta que los datos den evidencia en contra |
| H₁ | Hipótesis alternativa | Lo que se quiere demostrar |
| α | Nivel de significancia | Probabilidad máxima tolerada de **Error Tipo I**. Típicamente α = 0,05 |
| 1−α | Nivel de confianza (NC) | Probabilidad de no rechazar H₀ cuando es verdadera. Ej.: α = 0,05 → NC = 95% |
| p-valor | Probabilidad del estadístico | Prob. de observar un estadístico igual o más extremo suponiendo H₀ verdadera |
| n | Tamaño de muestra | Número de observaciones |
| x̄ | Media muestral | Promedio aritmético de la muestra |
| S | Desvío estándar muestral | Raíz cuadrada de la varianza muestral con divisor n−1 (`ddof=1`) |
| S² | Varianza muestral | Cuadrado de S |

### Tipos de error (tabla de decisiones)

| | H₀ **verdadera** | H₀ **falsa** |
|---|---|---|
| **No rechazar H₀** | ✅ Decisión correcta (NC = 1−α) | ❌ Error Tipo II (prob. β) |
| **Rechazar H₀** | ❌ Error Tipo I (prob. α) | ✅ Decisión correcta (potencia = 1−β) |

### Ejemplos de los cuatro casos

Los cuatro casos se ilustran con tres dominios. En cada uno el enunciado establece una H₀ y se muestran las cuatro combinaciones posibles entre **realidad** y **decisión estadística**.

---

#### 🏥 Medicina — ¿Un medicamento reduce la presión arterial?
H₀: el medicamento **no** reduce la presión arterial (no tiene efecto).

| Caso | Realidad | Decisión del test | Consecuencia |
|---|---|---|---|
| ✅ Correcto (NC = 1−α) | El medicamento no tiene efecto | No se rechaza H₀ | Se descarta correctamente — no se aprueba un fármaco ineficaz |
| ❌ **Error Tipo I** (α) | El medicamento no tiene efecto | Se rechaza H₀ | Se aprueba un fármaco que en realidad no sirve — **falsa alarma** |
| ❌ **Error Tipo II** (β) | El medicamento sí tiene efecto | No se rechaza H₀ | Se descarta un fármaco que sí funcionaba — **oportunidad perdida** |
| ✅ Correcto (potencia = 1−β) | El medicamento sí tiene efecto | Se rechaza H₀ | Se aprueba un fármaco eficaz — detección correcta |

> En medicina, el **Error Tipo I** (aprobar un medicamento ineficaz) suele ser el más peligroso,
> por eso se elige α pequeño (1% o 5%).

---

#### 📈 Finanzas — ¿La rentabilidad media de una cartera supera el 8% anual?
H₀: la rentabilidad media es **igual o menor** al 8% anual.

| Caso | Realidad | Decisión del test | Consecuencia |
|---|---|---|---|
| ✅ Correcto (NC = 1−α) | La cartera no supera el 8% | No se rechaza H₀ | No se invierte — se evita una mala colocación de capital |
| ❌ **Error Tipo I** (α) | La cartera no supera el 8% | Se rechaza H₀ | Se invierte en una cartera que no cumple el objetivo — **pérdida de capital** |
| ❌ **Error Tipo II** (β) | La cartera sí supera el 8% | No se rechaza H₀ | Se pierde la oportunidad de invertir en una cartera rentable — **costo de oportunidad** |
| ✅ Correcto (potencia = 1−β) | La cartera sí supera el 8% | Se rechaza H₀ | Se invierte acertadamente en una cartera con buena rentabilidad |

> En finanzas ambos errores tienen costos concretos: el Tipo I implica pérdida directa,
> el Tipo II implica un costo de oportunidad. El balance entre α y β depende del perfil de riesgo.

---

#### 🏭 Control de calidad — ¿El peso promedio de un producto cumple el estándar de 500 g?
H₀: el peso promedio de producción **es igual** a 500 g.

| Caso | Realidad | Decisión del test | Consecuencia |
|---|---|---|---|
| ✅ Correcto (NC = 1−α) | La línea produce a 500 g | No se rechaza H₀ | La producción continúa — no se detiene una línea que funciona bien |
| ❌ **Error Tipo I** (α) | La línea produce a 500 g | Se rechaza H₀ | Se detiene la producción innecesariamente — **falsa alarma, costo operativo** |
| ❌ **Error Tipo II** (β) | La línea está descalibrada | No se rechaza H₀ | Se envían productos defectuosos al mercado — **reclamos, devoluciones** |
| ✅ Correcto (potencia = 1−β) | La línea está descalibrada | Se rechaza H₀ | Se detecta y corrige el problema antes de distribuir — detección correcta |

> En control de calidad, el **Error Tipo II** (no detectar un defecto) suele ser
> más costoso que el Tipo I (parar una línea por error).
> Por eso en este contexto se prioriza aumentar la potencia (1−β).

---

### Tipos de hipótesis alternativa

| Nombre | Notación | `simbolo_H1` (profesor) | `alternative` (scipy) |
|---|---|---|---|
| Unilateral izquierda | H₁: θ < θ₀ | `'<'` | `'less'` |
| Unilateral derecha | H₁: θ > θ₀ | `'>'` | `'greater'` |
| Bilateral | H₁: θ ≠ θ₀ | `'!='` | `'two-sided'` |

### Dos criterios de decisión equivalentes

**Por región crítica (RC):** se calcula un valor crítico θ_α a partir de α y
la distribución del estadístico. Si el estadístico observado cae en la RC, se rechaza H₀.

**Por p-valor:** se calcula la probabilidad de observar el estadístico bajo H₀.

> **Rechazar H₀ ⟺ p-valor < α**

> Ambos criterios siempre producen la **misma decisión** para el mismo α.
> Las funciones del profesor devuelven **ambos** resultados simultáneamente.

### Tabla resumen — los 6 tests paramétricos

| Test | Parámetro bajo H₀ | Estadístico | Distribución bajo H₀ | scipy / statsmodels | Función profesor |
|---|---|---|---|---|---|
| z — media (σ conocida) | μ = μ₀ | (x̄ − μ₀) / (σ₀/√n) | N(0,1) | `ss.norm.ppf/.cdf` | — |
| **t — media (σ desconocida)** | μ = μ₀ | (x̄ − μ₀) / (S/√n) | t(n−1) | `ss.ttest_1samp` | **`TH_MEDIA_VARDESCON`** |
| **χ² — varianza** | σ² = σ₀² | (n−1)·S² / σ₀² | χ²(n−1) | `ss.chi2.ppf/.cdf` | **`TH_MEDIA_VARIANZA`** |
| z — proporción | p = p₀ | (p̂ − p₀) / √(p₀·(1−p₀)/n) | N(0,1) | `proportions_ztest` | — |
| t — dif. medias (ind.) | μ₁ = μ₂ | pooled / Welch | t(n₁+n₂−2) / t(ν) | `ss.ttest_ind` | — |
| t — dif. medias (par.) | μd = 0 | d̄ / (Sd/√n) | t(n−1) | `ss.ttest_rel` | — |
| F — cociente varianzas | σ₁² = σ₂² | S₁² / S₂² | F(ν₁,ν₂) | `ss.f.ppf/.cdf` | — |

---
## 1 · Test z para la media (varianza poblacional conocida)

Se aplica cuando se conoce σ₀, la desviación estándar **poblacional**.
Es el caso base teórico; el t-test (sección 2) es su generalización para σ desconocida.

### Estadístico

> **z_obs = (x̄ − μ₀) / (σ₀ / √n)**

Bajo H₀, z_obs sigue una distribución **normal estándar** N(0,1).

### Región crítica y p-valor según H₁

| H₁ | Región crítica | p-valor | scipy |
|---|---|---|---|
| μ < μ₀ | z_obs < z_α | Φ(z_obs) | `ss.norm.cdf(z_obs)` |
| μ > μ₀ | z_obs > z_{1−α} | 1 − Φ(z_obs) | `1 - ss.norm.cdf(z_obs)` |
| μ ≠ μ₀ | \|z_obs\| > z_{1−α/2} | 2·(1 − Φ(\|z_obs\|)) | `2*(1 - ss.norm.cdf(abs(z_obs)))` |

donde z_α = Φ⁻¹(α) = `ss.norm.ppf(alfa)` y Φ = `ss.norm.cdf`.

> **scipy no tiene una función de alto nivel para este test.**
> Se implementa directamente con `ss.norm.ppf` (valor crítico) y `ss.norm.cdf` (p-valor).

In [3]:
def z_test_media(x_bar, mu0, sigma0, n, alfa, h1):
    """
    Test z para la media con varianza poblacional CONOCIDA.

    Parámetros
    ----------
    x_bar  : float — Media muestral (x̄)
    mu0    : float — Media poblacional bajo H0 (μ₀)
    sigma0 : float — Desvío estándar POBLACIONAL conocido (σ₀)
    n      : int   — Tamaño de muestra
    alfa   : float — Nivel de significancia α (ej. 0.05)
    h1     : str   — Dirección de H1: '<' | '>' | '!='

    Retorna
    -------
    dict con z_obs, z_crit, p_valor, decision

    Estadístico
    -----------
    z_obs = (x̄ - μ₀) / (σ₀ / √n)  ~  N(0,1)
    """
    z_obs = (x_bar - mu0) / (sigma0 / np.sqrt(n))

    if h1 == "<":
        z_crit  = ss.norm.ppf(alfa)                   # z_α  (cuantil cola izquierda)
        p_valor = ss.norm.cdf(z_obs)                  # P(Z ≤ z_obs)
        rechaza = z_obs < z_crit
    elif h1 == ">":
        z_crit  = ss.norm.ppf(1 - alfa)               # z_{1-α}
        p_valor = 1 - ss.norm.cdf(z_obs)              # P(Z > z_obs)
        rechaza = z_obs > z_crit
    else:                                              # bilateral
        z_crit  = ss.norm.ppf(1 - alfa / 2)           # z_{1-α/2}
        p_valor = 2 * (1 - ss.norm.cdf(abs(z_obs)))   # 2·P(Z > |z_obs|)
        rechaza = abs(z_obs) > z_crit

    return {
        "z_obs"   : round(z_obs, 4),
        "z_crit"  : round(z_crit, 4),
        "p_valor" : round(p_valor, 6),
        "decision": "RECHAZAR H0" if rechaza else "NO rechazar H0"
    }


# ── Ejemplo: ¿Es la media del Merval menor a $2.500.000?
# Se usa S como σ₀ hipotética (solo a efectos ilustrativos del z-test)
res = z_test_media(x_bar=media, mu0=2_500_000, sigma0=S, n=n, alfa=0.05, h1="<")

print("── TEST z PARA LA MEDIA (σ conocida) ─────────────────────────")
print(f"H0: μ ≥ 2.500.000   |   H1: μ < 2.500.000   |   α = 0.05")
print(f"z_obs    = {res['z_obs']}")
print(f"z_crit   = {res['z_crit']}  (z_α = Φ⁻¹(0.05), cola izquierda)")
print(f"p-valor  = {res['p_valor']}")
print(f"Decisión → {res['decision']}")

── TEST z PARA LA MEDIA (σ conocida) ─────────────────────────
H0: μ ≥ 2.500.000   |   H1: μ < 2.500.000   |   α = 0.05
z_obs    = -10.4185
z_crit   = -1.6449  (z_α = Φ⁻¹(0.05), cola izquierda)
p-valor  = 0.0
Decisión → RECHAZAR H0


---
## 2 · Test t para la media (varianza poblacional desconocida)

Caso más habitual en la práctica: no conocemos σ y la estimamos con el **desvío muestral** S.

### Estadístico

> **t_obs = (x̄ − μ₀) / (S / √n)**

Bajo H₀, sigue una distribución **t de Student** con n−1 grados de libertad.
A medida que n → ∞, la t(n−1) converge a N(0,1).

### Región crítica y p-valor según H₁

| H₁ | Región crítica | p-valor | scipy |
|---|---|---|---|
| μ < μ₀ | t_obs < t_{α; n−1} | F_t(t_obs; n−1) | `ss.t.cdf(t_obs, n-1)` |
| μ > μ₀ | t_obs > t_{1−α; n−1} | 1 − F_t(t_obs; n−1) | `1 - ss.t.cdf(t_obs, n-1)` |
| μ ≠ μ₀ | \|t_obs\| > t_{1−α/2; n−1} | 2·[1 − F_t(\|t_obs\|; n−1)] | `2*(1-ss.t.cdf(abs(t_obs), n-1))` |

donde t_{α; n−1} = `ss.t.ppf(alfa, n-1)` y F_t(·; n−1) = CDF de la t de Student con n−1 gl.

> **Referencia al profesor:** `TH_MEDIA_VARDESCON` en `TPAD_Clase_11.ipynb`
> implementa exactamente este test con `ss.t.ppf` y `ss.t.cdf`.

In [4]:
def TH_MEDIA_VARDESCON(media_muestral, mu, S, n, alfa, simbolo_H1):
    """
    Test t de Student para la media con varianza DESCONOCIDA.
    Implementación del profesor — TPAD_Clase_11.ipynb.

    Parámetros
    ----------
    media_muestral : float — Media de la muestra (x̄)
    mu             : float — Media poblacional bajo H0 (μ₀)
    S              : float — Desvío estándar muestral (ddof=1)
    n              : int   — Tamaño de muestra
    alfa           : float — Nivel de significancia α (ej. 0.05)
    simbolo_H1     : str   — Dirección de H1:
                             '<'  → H1: μ < μ₀  (unilateral izquierda)
                             '>'  → H1: μ > μ₀  (unilateral derecha)
                             '!=' → H1: μ ≠ μ₀  (bilateral)

    Retorna
    -------
    tuple[str, str]
        [0] Resultado por región crítica (t_obs vs t_critico)
        [1] Resultado por p-valor (p_valor vs alfa)

    Estadístico
    -----------
    t_obs = (x̄ - μ₀) / (S / √n)  ~  t(n-1)
    """
    tobs = round((media_muestral - mu) / (S / np.sqrt(n)), 4)

    if simbolo_H1 == ">":
        talfa   = round(ss.t.ppf(1 - alfa, n - 1), 4)
        p_valor = round(1 - ss.t.cdf(tobs, n - 1), 8)
        cumple  = tobs > talfa
        desc    = f"tobs={tobs} {'>' if cumple else '<='} t_{{1-α;n-1}}={talfa}"
    elif simbolo_H1 == "<":
        talfa   = round(ss.t.ppf(alfa, n - 1), 4)
        p_valor = round(ss.t.cdf(tobs, n - 1), 8)
        cumple  = tobs < talfa
        desc    = f"tobs={tobs} {'<' if cumple else '>='} t_{{α;n-1}}={talfa}"
    else:                                               # bilateral
        talfa   = round(ss.t.ppf(1 - alfa / 2, n - 1), 4)
        p_valor = round(2 * (1 - ss.t.cdf(abs(tobs), n - 1)), 8)
        cumple  = abs(tobs) > talfa
        desc    = f"|tobs|={abs(tobs)} {'>' if cumple else '<='} t_{{1-α/2;n-1}}={talfa}"

    accion  = "se rechaza" if cumple else "NO se rechaza"
    resp_rc = f"C.D {'cumplida' if cumple else 'NO cumplida'}: {desc} → {accion} H0"

    cumple_pv = p_valor < alfa
    resp_pv   = (f"p-valor={p_valor} {'<' if cumple_pv else '>='} α={alfa} "
                 f"→ {'se rechaza' if cumple_pv else 'NO se rechaza'} H0")

    return resp_rc, resp_pv


# ── Ejemplo (Actividad 11 — inciso J): ¿Es la media del Merval < $2.500.000?
rc, pv = TH_MEDIA_VARDESCON(
    media_muestral = media,
    mu             = 2_500_000,
    S              = S,
    n              = n,
    alfa           = 0.05,
    simbolo_H1     = "<"
)
print("── TH_MEDIA_VARDESCON (función del profesor) ──────────────────")
print(f"H0: μ ≥ 2.500.000   |   H1: μ < 2.500.000   |   α = 0.05")
print(f"Región crítica : {rc}")
print(f"p-valor        : {pv}")

── TH_MEDIA_VARDESCON (función del profesor) ──────────────────
H0: μ ≥ 2.500.000   |   H1: μ < 2.500.000   |   α = 0.05
Región crítica : C.D cumplida: tobs=-10.4185 < t_{α;n-1}=-1.6517 → se rechaza H0
p-valor        : p-valor=0.0 < α=0.05 → se rechaza H0


In [7]:
# ── Mismo test con scipy.stats.ttest_1samp ─────────────────────────
# Calcula internamente t_obs = (x̄ - popmean) / (S/√n)
# y ajusta el p-valor según 'alternative'.
#
# Parámetros principales de ttest_1samp
# --------------------------------------
#   a           : array/Series — datos de la muestra
#   popmean     : float        — μ₀ bajo H0
#   alternative : str          — 'less' | 'greater' | 'two-sided'
#
# Retorna: TtestResult con atributos .statistic y .pvalue

resultado = ss.ttest_1samp(
    a           = precios,
    popmean     = 2_500_000,
    alternative = "less"          # H1: μ < μ₀
)
alfa = 0.05
print("── scipy.stats.ttest_1samp ────────────────────────────────────")
print(f"H0: μ ≥ 2.500.000   |   H1: μ < 2.500.000   |   α = {alfa}")
print(f"t_obs   = {resultado.statistic:.4f}")
print(f"p-valor = {resultado.pvalue:.6f}")
print()
if resultado.pvalue < alfa:
    print("→ RECHAZAR H0: hay evidencia suficiente con 95% de confianza")
else:
    print("→ NO rechazar H0: no hay evidencia suficiente")

── scipy.stats.ttest_1samp ────────────────────────────────────
H0: μ ≥ 2.500.000   |   H1: μ < 2.500.000   |   α = 0.05
t_obs   = -10.4185
p-valor = 0.000000

→ RECHAZAR H0: hay evidencia suficiente con 95% de confianza


### Comparación: `TH_MEDIA_VARDESCON` (profesor) vs. `scipy.stats.ttest_1samp`

| Aspecto | `TH_MEDIA_VARDESCON` | `ttest_1samp` |
|---|---|---|
| **Estadístico t** | calculado explícitamente: (x̄−μ₀)/(S/√n) | calculado internamente |
| **Valor crítico** | devuelto en el texto | no expuesto |
| **p-valor (H₁: <)** | `ss.t.cdf(tobs, n-1)` | `.pvalue` con `alternative='less'` |
| **p-valor (H₁: >)** | `1 - ss.t.cdf(tobs, n-1)` | `.pvalue` con `alternative='greater'` |
| **p-valor bilateral** | `2*(1 - ss.t.cdf(abs(tobs), n-1))` | `.pvalue` con `alternative='two-sided'` |
| **Criterio RC** | ✅ siempre incluido | ❌ no incluido |
| **Criterio p-valor** | ✅ siempre incluido | ✅ el usuario decide |
| **Especificación de H₁** | `simbolo_H1`: `'<'` / `'>'` / `'!='` | `alternative`: `'less'` / `'greater'` / `'two-sided'` |
| **Entrada** | estadísticos pre-calculados (x̄, S, n) | array/Serie de datos crudos |
| **Salida** | texto interpretativo completo | `TtestResult(statistic, pvalue)` |

> Los valores numéricos (t_obs y p-valor) son **idénticos** en ambos enfoques.
> La diferencia es pedagógica: la función del profesor explicita cada paso.

---
## 3 · Test χ² para la varianza

Se usa para contrastar si la varianza poblacional es igual a un valor hipotético σ₀².
**Supuesto:** la variable tiene distribución normal.

### Estadístico

> **χ²_obs = (n−1) · S² / σ₀²**

Bajo H₀, sigue una distribución **chi-cuadrado** con n−1 grados de libertad.

### Región crítica y p-valor según H₁

| H₁ | Región crítica | p-valor | scipy |
|---|---|---|---|
| σ² < σ₀² | χ²_obs < χ²_{α; n−1} | F_χ²(χ²_obs; n−1) | `ss.chi2.cdf(c, n-1)` |
| σ² > σ₀² | χ²_obs > χ²_{1−α; n−1} | 1 − F_χ²(χ²_obs; n−1) | `1 - ss.chi2.cdf(c, n-1)` |
| σ² ≠ σ₀² | χ²_obs < χ²_{α/2} ó χ²_obs > χ²_{1−α/2} | 2·min(p_izq, p_der) | ver nota |

donde χ²_{α; n−1} = `ss.chi2.ppf(alfa, n-1)`.

> ⚠️ **Caso bilateral:** la distribución χ² es **asimétrica** (solo valores positivos),
> por lo que las dos colas no tienen el mismo ancho. El p-valor bilateral es
> 2·min(P(χ² ≤ χ²_obs), P(χ² ≥ χ²_obs)) — distinto al bilateral de la t o la z.

> **Referencia al profesor:** `TH_MEDIA_VARIANZA` en `TPAD_Clase_11.ipynb`
> implementa este test con `ss.chi2.ppf` y `ss.chi2.cdf`.
> scipy **no tiene** función de alto nivel equivalente a `ttest_1samp` para la varianza.

In [8]:
def TH_MEDIA_VARIANZA(var_muestral, var_pob, n, alfa, simbolo_H1):
    """
    Test chi-cuadrado para la varianza poblacional.
    Implementación del profesor — TPAD_Clase_11.ipynb.

    Parámetros
    ----------
    var_muestral : float — Varianza muestral S² (ddof=1, como la devuelve pandas .var())
    var_pob      : float — Varianza poblacional bajo H0 (σ₀²)
    n            : int   — Tamaño de muestra
    alfa         : float — Nivel de significancia α (ej. 0.05)
    simbolo_H1   : str   — Dirección de H1:
                           '<'  → H1: σ² < σ₀²  (unilateral izquierda)
                           '>'  → H1: σ² > σ₀²  (unilateral derecha)
                           '!=' → H1: σ² ≠ σ₀²  (bilateral)

    Retorna
    -------
    tuple[str, str]
        [0] Resultado por región crítica
        [1] Resultado por p-valor

    Estadístico
    -----------
    χ²_obs = (n-1) * S² / σ₀²  ~  χ²(n-1)

    Nota (caso bilateral)
    -----
    La distribución χ² es asimétrica → p-valor bilateral = 2·min(p_izq, p_der),
    distinto al bilateral de la t o z donde ambas colas son simétricas.
    """
    chiobs = round((n - 1) * var_muestral / var_pob, 4)

    if simbolo_H1 == ">":
        chialfa = round(ss.chi2.ppf(1 - alfa, n - 1), 4)
        p_valor = round(1 - ss.chi2.cdf(chiobs, n - 1), 8)
        cumple  = chiobs > chialfa
        desc    = f"chiobs={chiobs} {'>' if cumple else '<='} chi_{{1-α;n-1}}={chialfa}"
    elif simbolo_H1 == "<":
        chialfa = round(ss.chi2.ppf(alfa, n - 1), 4)
        p_valor = round(ss.chi2.cdf(chiobs, n - 1), 8)
        cumple  = chiobs < chialfa
        desc    = f"chiobs={chiobs} {'<' if cumple else '>='} chi_{{α;n-1}}={chialfa}"
    else:                                                  # bilateral
        chialfa  = round(ss.chi2.ppf(1 - alfa / 2, n - 1), 4)
        p_izq    = ss.chi2.cdf(chiobs, n - 1)
        p_der    = 1 - p_izq
        p_valor  = round(2 * min(p_izq, p_der), 8)        # bilateral asimétrico
        chi_bajo = ss.chi2.ppf(alfa / 2, n - 1)
        cumple   = chiobs > chialfa or chiobs < chi_bajo
        desc     = f"chiobs={chiobs} {'cae' if cumple else 'no cae'} en la RC bilateral"

    accion  = "se rechaza" if cumple else "NO se rechaza"
    resp_rc = f"C.D {'cumplida' if cumple else 'NO cumplida'}: {desc} → {accion} H0"

    cumple_pv = p_valor < alfa
    resp_pv   = (f"p-valor={p_valor} {'<' if cumple_pv else '>='} α={alfa} "
                 f"→ {'se rechaza' if cumple_pv else 'NO se rechaza'} H0")

    return resp_rc, resp_pv


# ── Ejemplo: ¿Es la varianza del Merval mayor que (1.000.000)²?
var0 = (1_000_000) ** 2   # σ₀² hipotética

rc, pv = TH_MEDIA_VARIANZA(
    var_muestral = float(precios.var()),
    var_pob      = var0,
    n            = n,
    alfa         = 0.05,
    simbolo_H1   = ">"
)
print("── TH_MEDIA_VARIANZA (función del profesor) ───────────────────")
print(f"H0: σ² ≤ {var0:.2e}   |   H1: σ² > {var0:.2e}   |   α = 0.05")
print(f"Región crítica : {rc}")
print(f"p-valor        : {pv}")

── TH_MEDIA_VARIANZA (función del profesor) ───────────────────
H0: σ² ≤ 1.00e+12   |   H1: σ² > 1.00e+12   |   α = 0.05
Región crítica : C.D NO cumplida: chiobs=23.8143 <= chi_{1-α;n-1}=257.7585 → NO se rechaza H0
p-valor        : p-valor=1.0 >= α=0.05 → NO se rechaza H0


In [9]:
# ── Mismo test con scipy.stats.chi2 (cálculo paso a paso) ──────────
# scipy no tiene función de alto nivel para el test de varianza.
# Se construye directamente con ss.chi2.ppf y ss.chi2.cdf,
# igual que internamente lo hace TH_MEDIA_VARIANZA.
#
# ss.chi2.ppf(q, df)  → cuantil q de la distribución χ²(df)  [valor crítico]
# ss.chi2.cdf(x, df)  → P(χ² ≤ x)                            [p-valor acumulado]

var_muestral = float(precios.var())    # S² con ddof=1 (pandas por defecto)
var0         = (1_000_000) ** 2
alfa         = 0.05
gl           = n - 1                   # grados de libertad

# Estadístico
chi2_obs = (gl * var_muestral) / var0

# Valor crítico (H1: >) — cuantil 1-α de χ²(gl)
chi2_crit = ss.chi2.ppf(1 - alfa, gl)

# p-valor (H1: >) — área a la derecha de chi2_obs
p_valor = 1 - ss.chi2.cdf(chi2_obs, gl)

print("── scipy.stats.chi2 (manual, paso a paso) ─────────────────────")
print(f"H0: σ² ≤ {var0:.2e}   |   H1: σ² > {var0:.2e}   |   α = {alfa}")
print(f"S²          = {var_muestral:.4e}")
print(f"χ²_obs      = {chi2_obs:.4f}")
print(f"χ²_crit     = {chi2_crit:.4f}  (χ²_{{1-α}}, gl={gl})")
print(f"p-valor     = {p_valor:.6f}")
print()
if p_valor < alfa:
    print("→ RECHAZAR H0: varianza significativamente mayor que σ₀²")
else:
    print("→ NO rechazar H0: no hay evidencia suficiente")

── scipy.stats.chi2 (manual, paso a paso) ─────────────────────
H0: σ² ≤ 1.00e+12   |   H1: σ² > 1.00e+12   |   α = 0.05
S²          = 1.0727e+11
χ²_obs      = 23.8143
χ²_crit     = 257.7585  (χ²_{1-α}, gl=222)
p-valor     = 1.000000

→ NO rechazar H0: no hay evidencia suficiente


### Comparación: `TH_MEDIA_VARIANZA` (profesor) vs. `scipy.stats.chi2`

| Aspecto | `TH_MEDIA_VARIANZA` | `scipy.stats.chi2` (manual) |
|---|---|---|
| **Estadístico χ²** | `chiobs = (n−1)·S²/σ₀²` explícito | ídem |
| **Valor crítico (H₁: >)** | `ss.chi2.ppf(1-alfa, n-1)` | `ss.chi2.ppf(1-alfa, gl)` |
| **p-valor (H₁: >)** | `1 - ss.chi2.cdf(chiobs, n-1)` | `1 - ss.chi2.cdf(chi2_obs, gl)` |
| **Bilateral** | `2·min(p_izq, p_der)` — correcto para distribución asimétrica | ídem |
| **Función scipy de alto nivel** | no existe | no existe |
| **Salida** | texto interpretativo | valores numéricos crudos |

> scipy **no tiene** una función equivalente a `ttest_1samp` para el test de varianza.
> Tanto el profesor como el cálculo manual usan `chi2.ppf` y `chi2.cdf` directamente.
> El código del profesor y el manual son funcionalmente equivalentes.

---
## 4 · Test z para la proporción

Se usa para contrastar si una proporción poblacional p es igual a un valor hipotético p₀.

**Condición de aplicabilidad (tamaño muestral mínimo):**

> n·p₀ ≥ 5  y  n·(1−p₀) ≥ 5

### Estadístico

> **z_obs = (p̂ − p₀) / √(p₀·(1−p₀)/n)**

donde p̂ = k/n es la proporción muestral (k = número de éxitos observados).
Bajo H₀, z_obs sigue **aproximadamente** N(0,1).

### Región crítica y p-valor según H₁

| H₁ | Región crítica | p-valor | scipy |
|---|---|---|---|
| p < p₀ | z_obs < z_α | Φ(z_obs) | `ss.norm.cdf(z_obs)` |
| p > p₀ | z_obs > z_{1−α} | 1 − Φ(z_obs) | `1 - ss.norm.cdf(z_obs)` |
| p ≠ p₀ | \|z_obs\| > z_{1−α/2} | 2·(1 − Φ(\|z_obs\|)) | `2*(1-ss.norm.cdf(abs(z_obs)))` |

> ⚠️ **El profesor no definió una función para este test.**
> Se implementa con `statsmodels.stats.proportion.proportions_ztest`
> o manualmente con `scipy.stats.norm`.

> **Ejemplo:** de los días del período 2025, ¿la proporción de días
> con precio superior a $2.000.000 es significativamente distinta del 50%?

In [10]:
# ── Test z para la proporción — statsmodels ─────────────────────────
# proportions_ztest(count, nobs, value, alternative)
#
# Parámetros
# ----------
#   count       : int   — número de éxitos observados (k)
#   nobs        : int   — tamaño de muestra (n)
#   value       : float — proporción hipotética bajo H0 (p₀)
#   alternative : str   — 'smaller' | 'larger' | 'two-sided'
#
# Retorna: (z_statistic, p_valor)

umbral = 2_000_000
k      = int((precios > umbral).sum())    # días con Precio > umbral
p_hat  = k / n                            # proporción muestral p̂
p0     = 0.50                             # H0: p = 50%
alfa   = 0.05

# Verificar condición de aplicabilidad antes de ejecutar el test
print(f"Condición n·p₀ ≥ 5     : {n*p0:.0f} ≥ 5   → {'✅' if n*p0 >= 5 else '❌'}")
print(f"Condición n·(1-p₀) ≥ 5 : {n*(1-p0):.0f} ≥ 5   → {'✅' if n*(1-p0) >= 5 else '❌'}")
print()

z_stat, p_valor = proportions_ztest(
    count       = k,
    nobs        = n,
    value       = p0,
    alternative = "two-sided"    # H1: p ≠ 0.50
)

print("── statsmodels: proportions_ztest ─────────────────────────────")
print(f"Umbral  = ${umbral:,}  →  k={k} días  →  p̂ = {p_hat:.4f}")
print(f"H0: p = {p0}   |   H1: p ≠ {p0}   |   α = {alfa}")
print(f"z_obs   = {z_stat:.4f}")
print(f"p-valor = {p_valor:.6f}")
print()
if p_valor < alfa:
    print("→ RECHAZAR H0: la proporción difiere significativamente del 50%")
else:
    print("→ NO rechazar H0: no hay evidencia de diferencia significativa")

Condición n·p₀ ≥ 5     : 112 ≥ 5   → ✅
Condición n·(1-p₀) ≥ 5 : 112 ≥ 5   → ✅

── statsmodels: proportions_ztest ─────────────────────────────
Umbral  = $2,000,000  →  k=182 días  →  p̂ = 0.8161
H0: p = 0.5   |   H1: p ≠ 0.5   |   α = 0.05
z_obs   = 12.1875
p-valor = 0.000000

→ RECHAZAR H0: la proporción difiere significativamente del 50%


In [11]:
# ── Equivalente manual con scipy.stats.norm ─────────────────────────
# Construimos el estadístico paso a paso para trazar la fórmula.
# Los resultados deben coincidir con proportions_ztest.
#
# Fórmula: z_obs = (p̂ - p₀) / sqrt(p₀·(1-p₀)/n)

z_obs_manual = (p_hat - p0) / np.sqrt(p0 * (1 - p0) / n)

# Valor crítico bilateral: z_{1-α/2}
z_crit = ss.norm.ppf(1 - alfa / 2)

# p-valor bilateral: 2·P(Z > |z_obs|)
p_valor_manual = 2 * (1 - ss.norm.cdf(abs(z_obs_manual)))

print("── Cálculo manual con scipy.stats.norm ────────────────────────")
print(f"z_obs   = ({p_hat:.4f} - {p0}) / √({p0}·{1-p0}/{n})")
print(f"z_obs   = {z_obs_manual:.4f}")
print(f"z_crit  = ±{z_crit:.4f}  (z_{{1-α/2}} para α={alfa})")
print(f"p-valor = {p_valor_manual:.6f}")
print()
# Verificar que coincide con proportions_ztest
coincide = abs(z_obs_manual - z_stat) < 1e-6
print(f"Coincide con proportions_ztest: {'✅' if coincide else '❌'}")

── Cálculo manual con scipy.stats.norm ────────────────────────
z_obs   = (0.8161 - 0.5) / √(0.5·0.5/223)
z_obs   = 9.4421
z_crit  = ±1.9600  (z_{1-α/2} para α=0.05)
p-valor = 0.000000

Coincide con proportions_ztest: ❌


---
## 5 · Test t para diferencia de medias — dos muestras

### 5a. Muestras independientes

H₀: μ₁ − μ₂ = 0  (las dos poblaciones tienen la misma media)

#### Caso 1 — Varianzas iguales (estadístico pooled, asume σ₁² = σ₂²)

Se estima una varianza combinada ponderada:

> Sp² = [(n₁−1)·S₁² + (n₂−1)·S₂²] / (n₁+n₂−2)

> **t_obs = (x̄₁ − x̄₂) / (Sp · √(1/n₁ + 1/n₂))  ~  t(n₁+n₂−2)**

#### Caso 2 — Varianzas distintas (corrección de Welch — **recomendado por defecto**)

> **t_obs = (x̄₁ − x̄₂) / √(S₁²/n₁ + S₂²/n₂)  ~  t(ν)**

Los grados de libertad ν se estiman con la **fórmula de Satterthwaite**:

> ν = (S₁²/n₁ + S₂²/n₂)² / [(S₁²/n₁)²/(n₁−1) + (S₂²/n₂)²/(n₂−1)]

| Parámetro | Estadístico | g.d.l. |
|---|---|---|
| `equal_var=True` | t pooled | n₁ + n₂ − 2 (exacto) |
| `equal_var=False` | t Welch | ν (Satterthwaite, aproximado) |

> ⚠️ **El profesor no definió una función para este test.**
> Ante la duda sobre homocedasticidad, usar Welch (`equal_var=False`).
> Se puede verificar con el Test F (sección 6) o `ss.levene`.

> **Ejemplo:** ¿Difieren las medias del Merval entre el primer semestre
> (ene–jun) y el segundo (jul–nov)?

In [12]:
# ── Test t para muestras independientes — scipy.stats.ttest_ind ─────
# ttest_ind(a, b, equal_var, alternative)
#
# Parámetros
# ----------
#   a, b        : arrays/Series de las dos muestras independientes
#   equal_var   : True  → estadístico pooled  (asume σ₁² = σ₂²)
#                 False → corrección de Welch  (no asume igualdad)
#   alternative : 'less' | 'greater' | 'two-sided'
#
# Retorna: TtestResult con .statistic y .pvalue

alfa = 0.05
print(f"Semestre 1 : n={len(sem1):3d},  x̄=${sem1.mean():>13,.0f},  S=${sem1.std():>12,.0f}")
print(f"Semestre 2 : n={len(sem2):3d},  x̄=${sem2.mean():>13,.0f},  S=${sem2.std():>12,.0f}")
print(f"\nH0: μ₁ = μ₂   |   H1: μ₁ ≠ μ₂   |   α = {alfa}\n")

# ── Caso 1: varianzas iguales (pooled)
res_pool = ss.ttest_ind(sem1, sem2, equal_var=True,  alternative="two-sided")
print("[equal_var=True  — pooled, asume σ₁² = σ₂²]")
print(f"  t_obs   = {res_pool.statistic:.4f}")
print(f"  p-valor = {res_pool.pvalue:.6f}")
print(f"  Decisión → {'RECHAZAR H0' if res_pool.pvalue < alfa else 'NO rechazar H0'}")
print()

# ── Caso 2: varianzas distintas (Welch) — más robusto y recomendado
res_welch = ss.ttest_ind(sem1, sem2, equal_var=False, alternative="two-sided")
print("[equal_var=False — Welch, no asume igualdad de varianzas  ← recomendado]")
print(f"  t_obs   = {res_welch.statistic:.4f}")
print(f"  p-valor = {res_welch.pvalue:.6f}")
print(f"  Decisión → {'RECHAZAR H0' if res_welch.pvalue < alfa else 'NO rechazar H0'}")

Semestre 1 : n=118,  x̄=$    2,322,569,  S=$     204,773
Semestre 2 : n=105,  x̄=$    2,214,099,  S=$     418,956

H0: μ₁ = μ₂   |   H1: μ₁ ≠ μ₂   |   α = 0.05

[equal_var=True  — pooled, asume σ₁² = σ₂²]
  t_obs   = 2.4975
  p-valor = 0.013235
  Decisión → RECHAZAR H0

[equal_var=False — Welch, no asume igualdad de varianzas  ← recomendado]
  t_obs   = 2.4092
  p-valor = 0.017223
  Decisión → RECHAZAR H0


### 5b. Muestras pareadas (dependientes)

Se usa cuando cada observación de la muestra 1 está **emparejada** con una de la muestra 2
(mismo sujeto antes/después de un tratamiento; mismo activo en dos períodos consecutivos, etc.).

Se trabaja con las **diferencias** dᵢ = x₁ᵢ − x₂ᵢ:

> d̄ = (1/n)·Σdᵢ     Sd = √[Σ(dᵢ − d̄)² / (n−1)]

> **t_obs = d̄ / (Sd / √n)  ~  t(n−1)**

| H₁ | Región crítica | p-valor |
|---|---|---|
| μd < 0 | t_obs < t_{α; n−1} | `ss.t.cdf(t_obs, n-1)` |
| μd > 0 | t_obs > t_{1−α; n−1} | `1 - ss.t.cdf(t_obs, n-1)` |
| μd ≠ 0 | \|t_obs\| > t_{1−α/2; n−1} | `2*(1 - ss.t.cdf(abs(t_obs), n-1))` |

> **Ejemplo:** ¿Existe diferencia sistemática entre el precio del Merval
> en días de posición par vs. impar del período?

In [13]:
# ── Test t para muestras pareadas — scipy.stats.ttest_rel ───────────
# ttest_rel(a, b, alternative)
#
# Parámetros
# ----------
#   a, b        : arrays/Series de IGUAL longitud (observaciones emparejadas)
#   alternative : 'less' | 'greater' | 'two-sided'
#
# Internamente calcula: d = a - b  →  t_obs = d̄ / (S_d / √n)
#
# Retorna: TtestResult con .statistic y .pvalue

# Construcción de pares: días de posición par vs. impar (mismo período)
precios_idx = precios.reset_index(drop=True)
dias_par    = precios_idx.iloc[::2].reset_index(drop=True)    # posiciones 0, 2, 4, …
dias_impar  = precios_idx.iloc[1::2].reset_index(drop=True)   # posiciones 1, 3, 5, …
min_n       = min(len(dias_par), len(dias_impar))
dias_par    = dias_par[:min_n]
dias_impar  = dias_impar[:min_n]

alfa    = 0.05
res_rel = ss.ttest_rel(dias_par, dias_impar, alternative="two-sided")

# Verificación manual de la fórmula
diffs    = dias_par - dias_impar
d_bar    = float(diffs.mean())
S_d      = float(diffs.std())
t_manual = d_bar / (S_d / np.sqrt(min_n))

print("── scipy.stats.ttest_rel (muestras pareadas) ──────────────────")
print(f"n parejas = {min_n}")
print(f"d̄         = ${d_bar:,.2f}   |   S_d = ${S_d:,.2f}")
print(f"H0: μd = 0   |   H1: μd ≠ 0   |   α = {alfa}")
print(f"t_obs (scipy)  = {res_rel.statistic:.4f}")
print(f"t_obs (manual) = {t_manual:.4f}  ← verifica la fórmula d̄/(S_d/√n)")
print(f"p-valor        = {res_rel.pvalue:.6f}")
print()
if res_rel.pvalue < alfa:
    print("→ RECHAZAR H0: hay diferencia sistemática entre días pares e impares")
else:
    print("→ NO rechazar H0: no hay diferencia sistemática significativa")

── scipy.stats.ttest_rel (muestras pareadas) ──────────────────
n parejas = 111
d̄         = $-10,875.85   |   S_d = $85,071.84
H0: μd = 0   |   H1: μd ≠ 0   |   α = 0.05
t_obs (scipy)  = -1.3469
t_obs (manual) = -1.3469  ← verifica la fórmula d̄/(S_d/√n)
p-valor        = 0.180777

→ NO rechazar H0: no hay diferencia sistemática significativa


---
## 6 · Test F para cociente de varianzas

Se usa para contrastar si dos poblaciones tienen la **misma varianza**:
H₀: σ₁² = σ₂² (equivalentemente, σ₁²/σ₂² = 1).

**Supuesto:** ambas muestras provienen de distribuciones normales.

### Estadístico

> **F_obs = S₁² / S₂²**

Bajo H₀, sigue una distribución **F de Snedecor** con ν₁ = n₁−1 y ν₂ = n₂−1 grados de libertad.
Por convención se pone la varianza **mayor** en el numerador (F_obs ≥ 1).

### Región crítica y p-valor según H₁

| H₁ | Región crítica | p-valor |
|---|---|---|
| σ₁² < σ₂² | F_obs < F_{α; ν₁,ν₂} | `ss.f.cdf(F_obs, ν₁, ν₂)` |
| σ₁² > σ₂² | F_obs > F_{1−α; ν₁,ν₂} | `1 - ss.f.cdf(F_obs, ν₁, ν₂)` |
| σ₁² ≠ σ₂² | F_obs < F_{α/2} ó > F_{1−α/2} | `2·min(p_izq, p_der)` |

donde F_{α; ν₁,ν₂} = `ss.f.ppf(alfa, ν₁, ν₂)`.

> **Uso típico:** verificar la suposición de homogeneidad de varianzas antes de aplicar
> el t-test pooled (sección 5a). Si se rechaza H₀, usar Welch en `ttest_ind`.

> ⚠️ **El Test F es sensible a la no-normalidad.** En la práctica se prefiere
> el **Test de Levene** (`ss.levene`) que es más robusto.

> **Ejemplo:** ¿La volatilidad del Merval (varianza de los precios) difiere
> entre el primer y el segundo semestre del período?

In [14]:
def test_F_varianzas(muestra1, muestra2, alfa, h1):
    """
    Test F de Fisher para comparar varianzas de dos muestras independientes.
    scipy no tiene función de alto nivel; se construye con ss.f.ppf y ss.f.cdf.

    Parámetros
    ----------
    muestra1 : array-like — Primera muestra
    muestra2 : array-like — Segunda muestra
    alfa     : float      — Nivel de significancia α
    h1       : str        — Dirección de H1: '<' | '>' | '!='

    Retorna
    -------
    dict con S1_sq, S2_sq, F_obs, F_crit, p_valor, decision

    Estadístico
    -----------
    F_obs = S₁² / S₂²  ~  F(ν₁=n₁-1, ν₂=n₂-1)

    Supuesto
    --------
    El test F es sensible a la normalidad de las distribuciones.
    Para datos no normales o con outliers, preferir ss.levene (ver celda siguiente).
    """
    s1_sq = float(np.var(muestra1, ddof=1))   # varianza muestral S₁² (ddof=1)
    s2_sq = float(np.var(muestra2, ddof=1))   # varianza muestral S₂² (ddof=1)
    nu1   = len(muestra1) - 1                  # grados de libertad grupo 1
    nu2   = len(muestra2) - 1                  # grados de libertad grupo 2
    F_obs = s1_sq / s2_sq                      # estadístico observado

    if h1 == ">":
        F_crit  = ss.f.ppf(1 - alfa, nu1, nu2)        # F_{1-α; ν1,ν2}
        p_valor = 1 - ss.f.cdf(F_obs, nu1, nu2)       # P(F > F_obs)
        rechaza = F_obs > F_crit
    elif h1 == "<":
        F_crit  = ss.f.ppf(alfa, nu1, nu2)             # F_{α; ν1,ν2}
        p_valor = ss.f.cdf(F_obs, nu1, nu2)            # P(F < F_obs)
        rechaza = F_obs < F_crit
    else:                                               # bilateral
        F_crit  = ss.f.ppf(1 - alfa / 2, nu1, nu2)   # F_{1-α/2}
        p_izq   = ss.f.cdf(F_obs, nu1, nu2)
        p_der   = 1 - p_izq
        p_valor = 2 * min(p_izq, p_der)               # bilateral asimétrico
        rechaza = (F_obs > F_crit or
                   F_obs < ss.f.ppf(alfa / 2, nu1, nu2))

    return {
        "S1_sq"   : round(s1_sq, 2),
        "S2_sq"   : round(s2_sq, 2),
        "F_obs"   : round(F_obs, 4),
        "F_crit"  : round(F_crit, 4),
        "p_valor" : round(p_valor, 6),
        "decision": "RECHAZAR H0" if rechaza else "NO rechazar H0",
        "nu1"     : nu1,
        "nu2"     : nu2,
    }


# ── Ejemplo: ¿Son iguales las varianzas en sem1 y sem2?
res = test_F_varianzas(sem1, sem2, alfa=0.05, h1="!=")

print("── TEST F PARA COCIENTE DE VARIANZAS ──────────────────────────")
print(f"H0: σ₁² = σ₂²   |   H1: σ₁² ≠ σ₂²   |   α = 0.05")
print(f"S₁² (sem1) = {res['S1_sq']:>15,.0f}")
print(f"S₂² (sem2) = {res['S2_sq']:>15,.0f}")
print(f"F_obs      = {res['F_obs']}")
print(f"F_crit     = {res['F_crit']}  (F_{{1-α/2; {res['nu1']},{res['nu2']}}})")
print(f"p-valor    = {res['p_valor']}")
print(f"Decisión   → {res['decision']}")

── TEST F PARA COCIENTE DE VARIANZAS ──────────────────────────
H0: σ₁² = σ₂²   |   H1: σ₁² ≠ σ₂²   |   α = 0.05
S₁² (sem1) =  41,932,174,033
S₂² (sem2) = 175,524,248,614
F_obs      = 0.2389
F_crit     = 1.4588  (F_{1-α/2; 117,104})
p-valor    = 0.0
Decisión   → RECHAZAR H0


In [15]:
# ── Alternativa robusta: scipy.stats.levene ─────────────────────────
# Prueba de Levene para igualdad de varianzas.
# Ventaja frente al Test F: no asume normalidad; más resistente a outliers.
#
# Parámetros
# ----------
#   *args   : dos o más arrays/Series a comparar
#   center  : 'mean'   → Levene original
#             'median' → Brown-Forsythe (más robusto, recomendado)
#
# H0: σ₁² = σ₂²  (siempre bilateral en este test)
# Retorna: (W_statistic, p_valor)

W_stat, p_lev = ss.levene(sem1, sem2, center="median")
alfa = 0.05

print("── scipy.stats.levene (alternativa robusta al Test F) ─────────")
print(f"H0: σ₁² = σ₂²  |  H1: σ₁² ≠ σ₂²  |  α = {alfa}")
print(f"Estadístico W = {W_stat:.4f}")
print(f"p-valor       = {p_lev:.6f}")
print()
if p_lev < alfa:
    print("→ RECHAZAR H0: varianzas significativamente distintas")
    print("   → usar ttest_ind(equal_var=False)  [Welch]")
else:
    print("→ NO rechazar H0: no hay evidencia de varianzas distintas")
    print("   → se puede usar ttest_ind(equal_var=True)  [pooled]")

── scipy.stats.levene (alternativa robusta al Test F) ─────────
H0: σ₁² = σ₂²  |  H1: σ₁² ≠ σ₂²  |  α = 0.05
Estadístico W = 23.4779
p-valor       = 0.000002

→ RECHAZAR H0: varianzas significativamente distintas
   → usar ttest_ind(equal_var=False)  [Welch]


---
## Tabla de decisión final — ¿qué test aplicar?

| Pregunta de investigación | Parámetro | Test | Función |
|---|---|---|---|
| ¿La media es igual a μ₀? (σ conocida) | μ | z-test | `ss.norm` |
| ¿La media es igual a μ₀? (σ desconocida) | μ | t-test 1 muestra | `ttest_1samp` / `TH_MEDIA_VARDESCON` |
| ¿La varianza es igual a σ₀²? | σ² | χ²-test | `ss.chi2` / `TH_MEDIA_VARIANZA` |
| ¿La proporción es igual a p₀? | p | z-test proporción | `proportions_ztest` |
| ¿Dos medias independientes son iguales? (varianzas iguales) | μ₁−μ₂ | t-test 2 muestras pooled | `ttest_ind(equal_var=True)` |
| ¿Dos medias independientes son iguales? (varianzas distintas o desconocidas) | μ₁−μ₂ | t-test Welch | `ttest_ind(equal_var=False)` |
| ¿Dos medias pareadas son iguales? | μd | t-test pareado | `ttest_rel` |
| ¿Dos varianzas son iguales? | σ₁²/σ₂² | Test F | `ss.f` |
| ¿Hay homocedasticidad? (robusto a no-normalidad) | — | Test de Levene | `ss.levene` |

### Guía rápida de selección

```
¿Cuántos parámetros se prueban?
├── Uno
│   ├── ¿Es una media?
│   │   ├── σ conocida → z-test (secc. 1)
│   │   └── σ desconocida → t-test 1 muestra (secc. 2)  ← caso más común
│   ├── ¿Es una varianza? → χ²-test (secc. 3)
│   └── ¿Es una proporción? → z-test proporción (secc. 4)
└── Dos
    ├── ¿Son dos medias?
    │   ├── Muestras independientes → t-test 2 muestras (secc. 5a)
    │   │   ├── ¿Misma varianza? → pooled  ─ equal_var=True
    │   │   └── ¿Distinta varianza? → Welch ─ equal_var=False (recomendado por defecto)
    │   └── Muestras pareadas → t-test pareado (secc. 5b)
    └── ¿Son dos varianzas? → Test F (secc. 6)
```

> En la práctica, ante la duda sobre homocedasticidad, **usar Welch** y,
> si el supuesto de normalidad es cuestionable, **Test de Levene**.